# C03 · Q15 fixed point and timer math

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/BrandonZabawa/robotics-daily-labs/blob/main/labs/01_c_bare_metal/C03_q15_fixed_point_and_timer_math.ipynb)

**Track:** C and bare-metal · **Skill axis (sheet):** C and bare-metal · **Time:** about 50 min

**Skill Hub concepts:** Q15 fixed-point multiply; Timer prescaler and auto-reload

**Level 1** (the base exercises). Harder levels go at the end of this notebook.

**Languages:** each low-level task has an optional C/C++ cell after it. Same tests: write the cell, then uncomment the `lab.check(..., "file.c")` line under it. A check passes if either language passes.

> Cold attempt rules: notes closed, no AI, no internet until you run `lab.score()`. Fill every `TODO`, run the check under it, then do the checkpoint.

In [ ]:
#@title ⚙️ Setup: run this cell first
import os, sys, pathlib, subprocess
REPO_URL = "https://github.com/BrandonZabawa/robotics-daily-labs"   # or your fork
BRANCH = "main"                                  # the branch that holds your added levels
_root = os.environ.get("RDL_ROOT")
_root = pathlib.Path(_root) if _root else next((p for p in [pathlib.Path.cwd(), *pathlib.Path.cwd().parents] if (p / "roboprep").is_dir()), None)
if _root is None:
    _root = pathlib.Path("/content/robotics-daily-labs")
    if _root.exists():
        subprocess.run(["git", "-C", str(_root), "fetch", "-q", "origin", BRANCH])
        subprocess.run(["git", "-C", str(_root), "checkout", "-q", BRANCH])
        subprocess.run(["git", "-C", str(_root), "pull", "-q", "origin", BRANCH])
    else:
        subprocess.run(["git", "clone", "-q", "--depth", "1", "--branch", BRANCH, REPO_URL, str(_root)], check=True)
if str(_root) not in sys.path:
    sys.path.insert(0, str(_root))
from roboprep import Lab
lab = Lab("C03")

## Map: fixed-point formats and the timer equation

| Format | Stored as | Value | Range | Step |
|---|---|---|---|---|
| Q15 | `int16_t` | $x = n / 2^{15}$ | $[-1,\ 1-2^{-15}]$ | $3.05\times10^{-5}$ |
| Q31 | `int32_t` | $x = n / 2^{31}$ | $[-1,\ 1-2^{-31}]$ | $4.66\times10^{-10}$ |
| Q16.16 | `int32_t` | $x = n / 2^{16}$ | $[-32768,\ 32768)$ | $1.53\times10^{-5}$ |

Q15 multiply: the product of two Q15 numbers is Q30 in 32 bits; shift back by 15 with rounding, and saturate the one
case that overflows, $(-1)\times(-1) = +1$:
$$p = \mathrm{sat}_{16}\!\left(\frac{a\,b + 2^{14}}{2^{15}}\right)$$

Timer (STM32 style, counter runs 0 to ARR):
$$f_{\text{update}} = \frac{f_{\text{clk}}}{(\mathrm{PSC}+1)(\mathrm{ARR}+1)}, \qquad
\text{duty} = \frac{\mathrm{CCR}}{\mathrm{ARR}+1}, \qquad \text{PWM steps} = \mathrm{ARR}+1$$

| Choice | Effect |
|---|---|
| Small PSC, large ARR | fine duty resolution, short maximum period |
| Large PSC, small ARR | long periods, coarse resolution |

## Task 1: Q15 in C (rounding and saturation)

In [ ]:
%%writefile q15.c
#include <stdint.h>

int16_t q15_mul(int16_t a, int16_t b) {
    return (int16_t)((a * b) >> 15); /* TODO: round and saturate */
}

int16_t q15_from_float(float x) {
    return (int16_t)(x * 32768.0f); /* TODO: saturate and round */
}

In [ ]:
lab.check("q15")

## Task 2: pick PSC and ARR
Return `(psc, arr)` with both in `[0, 65535]` that hits `f_target` with the smallest frequency error; on a tie, the
smallest PSC (it keeps the most duty-cycle resolution).

In [ ]:
def timer_config(f_clk, f_target):
    raise NotImplementedError  # TODO

lab.check("timer", timer_config)

In [ ]:
%%writefile c03_timer.c
// C/C++ version of Task 2 (optional, same tests). Declarations: roboprep/c/include/c03.h
// Grade it with lab.check("timer", "c03_timer.c"). For C++, name the file .cpp in both places.
#include "c03.h"

void timer_config(double f_clk, double f_target, uint16_t *psc, uint16_t *arr) {
    (void)f_clk; (void)f_target;
    *psc = 0; *arr = 0;  /* TODO */
}

In [ ]:
# lab.check("timer", "c03_timer.c")  # uncomment after writing the C/C++ cell above

## Checkpoint: Math
**M1 (Apply).** $f_{\text{clk}} = 170$ MHz, PSC = 169, ARR = 999. Update frequency in Hz?

**M2 (Apply).** `q15_mul` of 0.5 and -0.25: the raw `int16_t` result?

**M3 (Analyze).** ARR = 999, CCR = 250. Duty cycle as a fraction?

In [ ]:
lab.answer("M1", None)
lab.answer("M2", None)
lab.answer("M3", None)

## Checkpoint: Concept
**K1 (Understand).** Why add $2^{14}$ before shifting right by 15?
- a) to avoid overflow
- b) so the result rounds to nearest instead of always toward negative infinity
- c) to convert from Q31 to Q15

**K2 (Evaluate).** For a 20 kHz motor PWM on a 170 MHz timer, pick PSC and ARR and defend the choice in terms of duty
resolution, and say why the formula has the "+1" terms.

In [ ]:
lab.choose("K1", None)

In [ ]:
lab.explain("K2", """
Your explanation here.
""")

In [ ]:
lab.self_score({"K2": 0})

## Level 2 and beyond: harder exercises you add

Add each new exercise above the score cell, in this order:

1. A markdown cell: what to build and why (a table or diagram if it helps).
2. A code cell with your stub (Python, or a `%%writefile` C/C++ cell that includes `c03.h`).
3. In `roboprep/checks/c03.py`, a grader: `def check_L2_<name>(fn): assert ...` (for C/C++ also
   `def c_L2_<name>(lib): return (callable, ...)` built with `roboprep.cbind.Functions`). Every `check_` function in that
   file counts toward the code score automatically.
4. `lab.check("L2_<name>", your_function)` (or the C/C++ file name) under your stub.

Commit the notebook and the grader to your branch and set `BRANCH` in the setup cell so Colab clones it.

In [ ]:
# Level 2: your exercises go here (see the cell above)

In [ ]:
lab.score()